# Identificación de especie con Pl@ntNet

Sugiere especie para las fotos de árboles/orquídeas/flores que quedaron en confianza "media" tras el escaneo del volumen (nombre común sin binomial, o carpetas administrativas como "Fichas"/"Articulos").

**No cubre aves ni insectos**, Pl@ntNet es solo plantas. Para esas ramas sigue el plan de iNaturalist (pendiente del profesor) o revisión manual.

**Insumo:** `../../experimentos/bd/campuslab_volumen.db` (no se mueve ni se duplica, se lee desde donde ya vive).
**Salida:** actualiza `especies.sugerido_por = 'PlantNet'` y dos columnas nuevas de auditoría (`plantnet_score`, `plantnet_raw`), nunca marca `fuente_validacion` (eso solo lo pone una confirmación humana/autorizada, no una sugerencia automática).

In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import requests
from io import BytesIO
from PIL import Image

def cargar_config(archivo="config_local.json"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontro {archivo}. Copia config_local.example.json a {archivo} "
            "(junto a este notebook) y pon tu API key de Pl@ntNet. No se sube a git."
        )
    return json.loads(ruta.read_text(encoding="utf-8"))

config = cargar_config()
PLANTNET_API_KEY = config["plantnet_api_key"]
RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")  # la BD real, sin mover ni duplicar

print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")

Base de datos: C:\Users\hecto\OneDrive\Escritorio\ITESO\Comunidad\Campus Lab\experimentos\bd\campuslab_volumen.db (existe)


In [2]:
con = sqlite3.connect(RUTA_BD)

# Columnas de auditoria para no perder el detalle de que sugirio Pl@ntNet y con que confianza
cols = [r[1] for r in con.execute("PRAGMA table_info(especies)")]
for col, tipo in [("plantnet_score", "REAL"), ("plantnet_raw", "TEXT")]:
    if col not in cols:
        con.execute(f"ALTER TABLE especies ADD COLUMN {col} {tipo}")

# Registro de TODO intento contra Pl@ntNet, aceptado o no. Sin esto, una corrida rechazada
# (score bajo o fuera de ESPECIES_OFICIALES) no queda marcada en ningun lado, y la siguiente
# corrida la vuelve a intentar y desperdicia cuota (limitada, diaria) en una foto que Pl@ntNet
# ya evaluo y va a evaluar exactamente igual otra vez.
con.execute("""
    CREATE TABLE IF NOT EXISTS plantnet_intentos (
        foto_id INTEGER PRIMARY KEY,
        nombre_sugerido TEXT,
        score REAL,
        aceptado INTEGER,
        fecha TEXT DEFAULT CURRENT_TIMESTAMP
    )
""")
con.commit()

# Candidatas: se filtraba antes por confianza_especie='media' de staging_escaneo (heuristica
# de carpeta), pero eso ignoraba el triage de Clasificador_PlantaAnimal.ipynb. Con
# reino_sugerido='planta' (CLIP) se cubre mucho mas: incluye las SINESP que nunca tuvieron
# ninguna pista de carpeta, y excluye fotos de animal/otro que antes se le mandaban a
# Pl@ntNet a desperdiciar cuota sin ninguna oportunidad de acertar.
# Se excluyen tambien las que ya estan en plantnet_intentos (resumible de verdad: no repite
# fotos ya intentadas, aceptadas o no).
consulta = """
    SELECT f.id AS foto_id, f.ruta_actual FROM fotos f
    WHERE f.reino_sugerido = 'planta' AND f.especie_id IS NULL AND f.ruta_actual NOT LIKE '%.db'
      AND f.id NOT IN (SELECT foto_id FROM plantnet_intentos)
"""
df_candidatas = pd.read_sql(consulta, con)

print(f"Total candidatas (reino_sugerido='planta', sin especie, sin intentar antes): {len(df_candidatas):,}")

# Lista oficial de especies de Campus Lab (BASE_DATOS_GENERAL.xlsx, ya espejada por
# Resolucion_Nombres.ipynb en referencia_nombres_comunes). Una sugerencia de Pl@ntNet
# solo se acepta si el nombre cae aqui: cierra la ventana a lo que realmente puede
# estar en Jalisco/ITESO, en vez de dejar que Pl@ntNet proponga cualquier especie del
# mundo con score decente pero geograficamente imposible.
ESPECIES_OFICIALES = {
    r[0] for r in con.execute(
        "SELECT DISTINCT nombre_cientifico FROM referencia_nombres_comunes WHERE nombre_cientifico IS NOT NULL"
    )
}
print(f"Especies oficiales cargadas (BASE_DATOS_GENERAL): {len(ESPECIES_OFICIALES):,}")

Total candidatas (reino_sugerido='planta', sin especie, sin intentar antes): 1,626
Especies oficiales cargadas (BASE_DATOS_GENERAL): 465


## Modo offline: espejo local de las candidatas

Para trabajar fuera de la red de ITESO (`Y:\Galería` no es alcanzable desde fuera). Copia cada foto candidata a `ruta_espejo_local_fotos` (definida en `config_local.json`), en una carpeta plana fuera del repo. `identificar_planta` ya busca ahí primero automáticamente (ver `resolver_ruta_legible` arriba); si el espejo no existe o la foto no se ha copiado, sigue usando `Y:\` normal, así que esta celda es opcional y nunca rompe el flujo de trabajo en ITESO.

**Resumible**: solo copia lo que falta o cambió de tamaño, se puede volver a correr sin duplicar trabajo.

In [ ]:
import shutil

RUTA_ESPEJO_LOCAL = config.get("ruta_espejo_local_fotos")
COPIAR_ESPEJO_LOCAL_REAL = False  # <-- cambialo a True para copiar de verdad

if not RUTA_ESPEJO_LOCAL:
    print("No hay 'ruta_espejo_local_fotos' en config_local.json. Se sigue leyendo directo de Y:\\ (modo normal en ITESO).")
else:
    destino = Path(RUTA_ESPEJO_LOCAL)
    destino.mkdir(parents=True, exist_ok=True)

    faltantes = []
    for ruta_str in df_candidatas["ruta_actual"]:
        origen = Path(ruta_str)
        local = destino / origen.name
        if not local.exists() or local.stat().st_size != origen.stat().st_size:
            faltantes.append((origen, local))

    tamano_gb = sum(o.stat().st_size for o, _ in faltantes) / 1024**3
    print(f"Espejo local: {destino}")
    print(f"Candidatas totales: {len(df_candidatas):,} | Ya copiadas: {len(df_candidatas) - len(faltantes):,} | Faltan: {len(faltantes):,} ({tamano_gb:.2f} GB)")

    if not COPIAR_ESPEJO_LOCAL_REAL:
        print("COPIAR_ESPEJO_LOCAL_REAL esta en False. No se copio nada todavia.")
    else:
        for i, (origen, local) in enumerate(faltantes, 1):
            shutil.copy2(origen, local)
            if i % 100 == 0 or i == len(faltantes):
                print(f"  {i}/{len(faltantes)} copiadas...")
        print(f"Listo: {len(faltantes)} fotos copiadas a {destino}")

## Función de identificación

Sube hasta 5 imágenes por solicitud (aquí mandamos de 1 en 1, más simple de auditar por foto). Según la documentación, la respuesta trae `results`, una lista ordenada por `score` con `species.scientificNameWithoutAuthor`, `species.family`, etc.

In [ ]:
EXTENSIONES_SOPORTADAS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
LADO_MAXIMO_PX = 1024  # de sobra para identificacion, mantiene el archivo enviado chico

def resolver_ruta_legible(ruta_actual):
    """Si hay un espejo local configurado (trabajo fuera de la red de ITESO, ver
    'ruta_espejo_local_fotos' en config_local.json) y la foto ya se copio ahi, usa esa
    copia. Si no hay espejo o la foto no esta ahi todavia, usa la ruta real en Y:\\
    (modo normal, dentro de la red de ITESO)."""
    ruta_espejo = config.get("ruta_espejo_local_fotos")
    if ruta_espejo:
        local = Path(ruta_espejo) / Path(ruta_actual).name
        if local.exists():
            return local
    return Path(ruta_actual)

def preparar_imagen(ruta_imagen):
    """Abre, redimensiona si hace falta, y devuelve bytes JPEG listos para enviar.
    Devuelve None si el archivo no es un formato soportado (ej. RAW de camara, Thumbs.db)."""
    ruta_imagen = resolver_ruta_legible(ruta_imagen)
    ext = ruta_imagen.suffix.lower()
    if ext not in EXTENSIONES_SOPORTADAS:
        return None
    with Image.open(ruta_imagen) as img:
        img = img.convert("RGB")
        img.thumbnail((LADO_MAXIMO_PX, LADO_MAXIMO_PX))
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return buffer.getvalue()

def identificar_planta(ruta_imagen, organo="auto", api_key=PLANTNET_API_KEY):
    """Llama a Pl@ntNet con una sola imagen. Devuelve (nombre_cientifico, score, respuesta_completa).
    Si el archivo no es un formato soportado, devuelve (None, None, {"omitido": "formato no soportado"})
    sin gastar una llamada a la API."""
    imagen_bytes = preparar_imagen(ruta_imagen)
    if imagen_bytes is None:
        return None, None, {"omitido": f"formato no soportado: {Path(ruta_imagen).suffix}"}

    url = f"https://my-api.plantnet.org/v2/identify/all?api-key={api_key}"
    archivos = {"images": (Path(ruta_imagen).stem + ".jpg", imagen_bytes, "image/jpeg")}
    datos = {"organs": organo}
    resp = requests.post(url, files=archivos, data=datos, timeout=30)

    if not resp.ok:
        # Antes solo se veia el codigo (ej. "404 Client Error"). Esto guarda el mensaje real
        # del servidor, que normalmente explica la causa concreta.
        detalle = resp.text[:500]
        raise requests.HTTPError(f"{resp.status_code} {resp.reason}: {detalle}", response=resp)

    cuerpo = resp.json()
    if cuerpo.get("results"):
        mejor = cuerpo["results"][0]
        return mejor["species"]["scientificNameWithoutAuthor"], mejor["score"], cuerpo
    return None, None, cuerpo

## Prueba de parseo (sin llamar a la API real)

No tengo acceso de red a plantnet.org desde este entorno, así que antes de que la corras contra la API real, valido que el parseo de la respuesta funcione contra la forma exacta de JSON que documenta Pl@ntNet.

In [4]:
RESPUESTA_EJEMPLO_DOCUMENTACION = {
    "query": {"project": "all", "images": ["abc"], "organs": ["auto"]},
    "language": "en",
    "preferedReferential": "k-world-flora",
    "bestMatch": "Ajuga genevensis L.",
    "results": [
        {
            "score": 0.90734,
            "species": {
                "scientificNameWithoutAuthor": "Ajuga genevensis",
                "scientificNameAuthorship": "L.",
                "genus": {"scientificNameWithoutAuthor": "Ajuga"},
                "family": {"scientificNameWithoutAuthor": "Lamiaceae"},
            },
        }
    ],
}

def parsear_respuesta(cuerpo):
    if cuerpo.get("results"):
        mejor = cuerpo["results"][0]
        return mejor["species"]["scientificNameWithoutAuthor"], mejor["score"]
    return None, None

nombre, score = parsear_respuesta(RESPUESTA_EJEMPLO_DOCUMENTACION)
assert nombre == "Ajuga genevensis" and abs(score - 0.90734) < 1e-6
print(f"Parseo correcto: {nombre} (score {score})")

nombre_vacio, score_vacio = parsear_respuesta({"results": []})
assert nombre_vacio is None and score_vacio is None
print("Caso sin resultados manejado correctamente (no truena, regresa None)")

Parseo correcto: Ajuga genevensis (score 0.90734)
Caso sin resultados manejado correctamente (no truena, regresa None)


## Corrida real (intenta cubrir toda la cuota del día)

**No lo corras hasta tener tu API key en `config_local.json`.** Intenta identificar todas las candidatas restantes, pero se detiene solo cuando la cuota diaria de Pl@ntNet se agota (respuesta 429, o `remainingIdentificationRequests` llega a 0), nunca antes. Guarda avance cada 20 fotos, así que una corrida cortada a medias no pierde trabajo: la siguiente corrida retoma justo donde se quedó, sin repetir nada.

**Se acepta directo si `score >= SCORE_MINIMO`, o si `score >= UMBRAL_YA_CONFIRMADA` y esa especie ya está confirmada en el catálogo** (decisión de Remi, 11-sep: el filtro contra `ESPECIES_OFICIALES` ya no bloquea el auto-aceptar, solo el score importa). Las excepciones en `EXCEPCIONES_REVISAR_A_MANO` nunca se auto-aceptan sin importar el score. Si no pasa el filtro, la foto se queda sin `especie_id` pero el intento SÍ se registra en `plantnet_intentos`, así entra a la cola de revisión manual del revisor web con su score real.

In [ ]:
SCORE_MINIMO = 0.50  # Remi 11-sep: bajado de 0.70 a 0.50 despues de confirmar a mano que la calidad
                      # se mantiene hasta ahi (ajustable aqui)
UMBRAL_YA_CONFIRMADA = 0.35  # si la especie YA esta confirmada en el catalogo, un score mas bajo
                             # todavia es creible (ya sabemos que esa especie SI esta en el campus)
EJECUTAR_PILOTO_REAL = True

# 12-sep: ya no es un "piloto" chico, se intenta cubrir TODAS las candidatas en una corrida.
# El limite real ya no es este numero, es la cuota diaria de Pl@ntNet: el loop se detiene
# solo cuando la API contesta que se acabo (o remainingIdentificationRequests llega a 0),
# nunca sigue insistiendo y desperdiciando llamadas contra una cuota ya agotada.
TAMANO_PILOTO = len(df_candidatas)

# Casos donde NO conviene auto-aceptar aunque el score pase el umbral: Pl@ntNet lo sugirio 41
# veces distintas sin superar nunca 61%, y el propio nombre de carpeta original ya mostraba
# duda humana entre dos OTRAS especies ("Stanhopearadiosaosaccata" = "Stanhopea radiosa o
# saccata"). Las tres (radiosa/saccata/pseudoradiosa) son validas en GBIF, asi que no se puede
# desempatar solo con eso - necesita ojo experto. Si aparecen mas casos asi, se agregan aqui.
EXCEPCIONES_REVISAR_A_MANO = {"Stanhopea pseudoradiosa"}

especies_confirmadas = {r[0] for r in con.execute(
    "SELECT DISTINCT nombre_cientifico FROM especies e JOIN fotos f ON f.especie_id=e.id WHERE f.estado='usable'"
)}

if not EJECUTAR_PILOTO_REAL:
    print("EJECUTAR_PILOTO_REAL esta en False. No se llamo a la API. Pon tu API key y activalo cuando estes lista.")
else:
    muestra = df_candidatas.sample(min(TAMANO_PILOTO, len(df_candidatas)), random_state=1)
    resultados = []
    cuota_agotada = False
    for i, (_, fila) in enumerate(muestra.iterrows(), 1):
        try:
            nombre, score, crudo = identificar_planta(fila["ruta_actual"])
            # Auto-aceptar: score >= 50%, O ya sabemos que esa especie esta en el campus
            # (confirmada antes) y el score pasa un umbral mas bajo (35%). Nunca para las
            # excepciones marcadas arriba, sin importar el score.
            aceptado = (
                nombre is not None and score is not None and nombre not in EXCEPCIONES_REVISAR_A_MANO
                and (score >= SCORE_MINIMO or (nombre in especies_confirmadas and score >= UMBRAL_YA_CONFIRMADA))
            )
            resultados.append({
                "id": fila["foto_id"], "sugerido": nombre, "score": score, "error": None,
                "aceptado": aceptado, "_cuerpo_crudo": crudo,
            })
            # Se registra el intento SIEMPRE (aceptado o no), para que una corrida futura
            # nunca vuelva a mandar esta foto a Pl@ntNet y desperdicie cuota repitiendo
            # un resultado que ya conocemos.
            con.execute(
                "INSERT OR REPLACE INTO plantnet_intentos (foto_id, nombre_sugerido, score, aceptado) VALUES (?,?,?,?)",
                (int(fila["foto_id"]), nombre, score, int(aceptado)),
            )
            if aceptado:
                # Busca o crea la especie sugerida, y enlaza esta foto especifica a ella.
                # Si no paso el filtro no se liga nada: se queda sin especie_id, pero el
                # intento SI queda registrado arriba para que aparezca en la cola de revision
                # manual del revisor web con su score real.
                especie_row = con.execute("SELECT id FROM especies WHERE nombre_cientifico=?", (nombre,)).fetchone()
                if especie_row:
                    especie_id = especie_row[0]
                    con.execute(
                        "UPDATE especies SET sugerido_por=?, plantnet_score=?, plantnet_raw=? WHERE id=?",
                        ("PlantNet", score, json.dumps(crudo, ensure_ascii=False), especie_id)
                    )
                else:
                    cur = con.execute(
                        "INSERT INTO especies (nombre_cientifico, sugerido_por, plantnet_score, plantnet_raw) VALUES (?,?,?,?)",
                        (nombre, "PlantNet", score, json.dumps(crudo, ensure_ascii=False))
                    )
                    especie_id = cur.lastrowid
                    especies_confirmadas.add(nombre)  # para que rija tambien dentro de esta misma corrida
                con.execute("UPDATE fotos SET especie_id=?, estado=? WHERE id=?", (especie_id, "usable", fila["foto_id"]))

            # Commit seguido (no hasta el final): si la corrida se corta a medias (cuota,
            # corte de luz, lo que sea), no se pierde el trabajo ya hecho.
            if i % 20 == 0:
                con.commit()

            # Cuota real de Pl@ntNet: si esta foto fue la ultima que alcanzaba, no tiene caso
            # seguir intentando las demas (todas fallarian igual). Se detiene aqui mismo,
            # ya con esta foto procesada y guardada arriba.
            restante = crudo.get("remainingIdentificationRequests") if isinstance(crudo, dict) else None
            if restante is not None and restante <= 0:
                print(f"Cuota de Pl@ntNet en 0 despues de la foto {fila['foto_id']} ({i}/{len(muestra)}). Se detiene el lote aqui.")
                cuota_agotada = True
                con.commit()
                break
        except requests.HTTPError as e:
            resultados.append({"id": fila["foto_id"], "sugerido": None, "score": None, "error": str(e), "aceptado": False})
            # 429 = cuota agotada de verdad (no un error de esta foto en particular). Esta
            # foto especifica NO quedo en plantnet_intentos, asi que la proxima corrida la
            # vuelve a intentar sola, sin que nadie tenga que hacer nada manual.
            if e.response is not None and e.response.status_code == 429:
                print(f"Cuota de Pl@ntNet agotada (429) en foto {fila['foto_id']} ({i}/{len(muestra)}). Se detiene el lote aqui.")
                cuota_agotada = True
                con.commit()
                break
        except Exception as e:
            resultados.append({"id": fila["foto_id"], "sugerido": None, "score": None, "error": str(e), "aceptado": False})
    con.commit()

    df_resultado = pd.DataFrame(resultados)
    print(df_resultado.drop(columns=["_cuerpo_crudo"], errors="ignore").to_string(index=False))
    print(f"\nExitosos: {df_resultado['error'].isna().sum()} / {len(df_resultado)}")
    print(f"Aceptados (score >= {SCORE_MINIMO}, o >= {UMBRAL_YA_CONFIRMADA} si ya estaba confirmada): {df_resultado['aceptado'].sum()} / {len(df_resultado)}")
    if cuota_agotada:
        print(f"Quedaron {len(df_candidatas) - len(df_resultado)} candidatas sin intentar todavia, la cuota se acabo hoy. Vuelve a correr esta celda cuando se reponga.")

    # Cuota restante de Pl@ntNet, para saber si el lote siguiente debe ser mas chico
    ultimo_restante = None
    for r in reversed(resultados):
        crudo = r.get("_cuerpo_crudo")
        if isinstance(crudo, dict) and "remainingIdentificationRequests" in crudo:
            ultimo_restante = crudo["remainingIdentificationRequests"]
            break
    if ultimo_restante is not None:
        print(f"Solicitudes restantes hoy en tu cuota de Pl@ntNet: {ultimo_restante}")

# Nota: la conexion se deja abierta a proposito, para poder re-correr esta celda
# (reintentar el piloto, por ejemplo) sin tener que re-ejecutar todo el notebook desde el inicio.